In [0]:

# Source file location
orders_path = "/Volumes/workspace/ecom_poc/raw_files/orders.csv"

# Read the CSV using Apache Spark
orders_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(orders_path)
)

# Check the number of rows
print("Orders count:", orders_df.count())

# Check column names and data types
orders_df.printSchema()


In [0]:

# Path to the transactions source file
transactions_path = "/Volumes/workspace/ecom_poc/raw_files/transactions.csv"

# Read transactions as strings
transactions_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(transactions_path)
)

# Verify the row count
print("Transactions count:", transactions_df.count())

# Verify the schema
transactions_df.printSchema()


In [0]:

from pyspark.sql import functions as F

# Target table and batch identity
table_name = "workspace.ecom_poc.bronze_orders"
batch_id = "orders_seed42_v1"
expected_rows = 1_000_000

# Only the eight original source columns
source_columns = [
    "order_id", "customer_id", "order_date",
    "category", "quantity", "unit_price",
    "order_status", "country"
]

# Add metadata without modifying the source fields
bronze_orders_df = (
    orders_df
    .select(
        "*",
        F.col("_metadata.file_path").alias("_source_file")
    )
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_batch_id", F.lit(batch_id))
    .withColumn(
        "_row_hash",
        F.sha2(
            F.to_json(
                F.struct(*[F.col(c) for c in source_columns]),
                options={"ignoreNullFields": "false"}
            ),
            256
        )
    )
    .withColumn("_source_row_id", F.expr("uuid()"))
)

# Check whether this batch is already present
existing_rows = 0

if spark.catalog.tableExists(table_name):
    existing_rows = (
        spark.table(table_name)
        .filter(F.col("_batch_id") == batch_id)
        .count()
    )

if existing_rows == expected_rows:
    print("Batch already loaded. Skipping ingestion.")

elif existing_rows != 0:
    raise ValueError(
        f"Incomplete batch found: {existing_rows} rows. "
        "Investigate before retrying."
    )

else:
    (
        bronze_orders_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(table_name)
    )
    print("Bronze orders table created successfully!")

# Verify persisted records
print("Bronze orders rows:", spark.table(table_name).count())


In [0]:
orders_count = spark.table(
    "workspace.ecom_poc.bronze_orders"
).count()

print("Bronze orders:", orders_count)

assert orders_count == 1_000_000, "Orders count mismatch!"

In [0]:

from pyspark.sql import functions as F

# 1. Configuration
table_name = "workspace.ecom_poc.bronze_transactions"
batch_id = "transactions_seed2026_v1"
expected_rows = 1_000_000

source_columns = [
    "txn_id",
    "order_id",
    "payment_method",
    "amount",
    "currency",
    "txn_status",
    "txn_ts"
]

# 2. Add ingestion metadata
bronze_transactions_df = (
    transactions_df
    .select(
        "*",
        F.col("_metadata.file_path").alias("_source_file")
    )
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_batch_id", F.lit(batch_id))
    .withColumn(
        "_row_hash",
        F.sha2(
            F.to_json(
                F.struct(
                    *[F.col(c) for c in source_columns]
                ),
                options={"ignoreNullFields": "false"}
            ),
            256
        )
    )
    .withColumn("_source_row_id", F.expr("uuid()"))
)

# 3. Check for an existing ingestion batch
existing_rows = 0

if spark.catalog.tableExists(table_name):
    existing_rows = (
        spark.table(table_name)
        .filter(F.col("_batch_id") == batch_id)
        .count()
    )

# 4. Write only if the batch has not been loaded
if existing_rows == expected_rows:
    print("Batch already loaded. Skipping ingestion.")

elif existing_rows != 0:
    raise ValueError(
        f"Incomplete batch: {existing_rows} records"
    )

else:
    (
        bronze_transactions_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(table_name)
    )

    print("Bronze transactions table created!")

# 5. Verify the stored records
actual_rows = spark.table(table_name).count()

print("Bronze transactions rows:", actual_rows)

assert actual_rows == expected_rows, "Row count mismatch!"


In [0]:

from pyspark.sql import functions as F

datasets = [
    ("orders", "orders_seed42_v1"),
    ("transactions", "transactions_seed2026_v1")
]

for name, batch_id in datasets:

    df = spark.table(
        f"workspace.ecom_poc.bronze_{name}"
    )

    result = df.agg(
        F.count("*").alias("total_rows"),

        F.countDistinct("_source_row_id").alias("unique_ids"),

        F.sum(
            F.when(
                F.col("_source_row_id").isNull()
                | F.col("_ingested_at").isNull()
                | F.col("_source_file").isNull()
                | F.col("_batch_id").isNull()
                | (F.col("_batch_id") != batch_id)
                | F.col("_row_hash").isNull(),
                1
            ).otherwise(0)
        ).alias("audit_issues")
    ).first()

    print(f"\nDataset: {name}")
    print("Total rows:", result["total_rows"])
    print("Unique source IDs:", result["unique_ids"])
    print("Audit issues:", result["audit_issues"])

    assert result["total_rows"] == 1_000_000
    assert result["unique_ids"] == 1_000_000
    assert result["audit_issues"] == 0

    print("Verification: PASSED")


In [0]:

# Verify that rerunning did not duplicate records

orders_count = spark.table(
    "workspace.ecom_poc.bronze_orders"
).count()

transactions_count = spark.table(
    "workspace.ecom_poc.bronze_transactions"
).count()

print("Orders:", orders_count)
print("Transactions:", transactions_count)

assert orders_count == 1_000_000
assert transactions_count == 1_000_000

print("IDEMPOTENCY TEST: PASSED")


In [0]:

# Source-to-Bronze fidelity verification

datasets = {
    "orders": [
        "order_id", "customer_id", "order_date",
        "category", "quantity", "unit_price",
        "order_status", "country"
    ],
    "transactions": [
        "txn_id", "order_id", "payment_method",
        "amount", "currency", "txn_status", "txn_ts"
    ]
}

for name, columns in datasets.items():

    source_path = (
        f"/Volumes/workspace/ecom_poc/raw_files/{name}.csv"
    )

    # Read original CSV fields
    source_df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "false")
        .load(source_path)
        .select(*columns)
    )

    # Read only original fields from Bronze
    bronze_df = (
        spark.table(f"workspace.ecom_poc.bronze_{name}")
        .select(*columns)
    )

    # Compare in both directions, preserving duplicates
    missing_in_bronze = (
        source_df.exceptAll(bronze_df).count()
    )

    extra_in_bronze = (
        bronze_df.exceptAll(source_df).count()
    )

    print(f"\nDataset: {name}")
    print("Missing in Bronze:", missing_in_bronze)
    print("Extra in Bronze:", extra_in_bronze)

    assert missing_in_bronze == 0
    assert extra_in_bronze == 0

    print("SOURCE FIDELITY: PASSED")
